# M9 · Final results for the report

This notebook only reads the result files from earlier steps and makes the final tables and plots.

**Kaggle:** Accelerator None. **Inputs:** add the outputs of the **M3, M6, M7 and M8** notebooks.

In [ ]:
# ---- Setup: get the latest project code (run this cell first, every session) ----
import os, sys

REPO_URL = "https://github.com/YOUR-TEAM/moe-offload.git"   # <-- change to your team's repo
# Private repo? Add a Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets), then uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")

if os.path.exists("/kaggle"):                      # running on Kaggle
    import glob, shutil
    CODE_DIR = "/kaggle/working/moe-offload"
    # Option A: the code was uploaded as a Kaggle dataset ("moe-offload-code") and added as an input
    uploaded = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/moe-offload-code/config.py", recursive=True)]
    if uploaded:
        shutil.copytree(uploaded[0], CODE_DIR, dirs_exist_ok=True)
        print("using code from dataset:", uploaded[0])
    # Option B: get the code from GitHub
    elif not os.path.exists(CODE_DIR):
        !git clone -q {REPO_URL} {CODE_DIR}         # first time: download the code
    else:
        !git -C {CODE_DIR} pull -q                  # later: get the newest commits
    os.chdir(CODE_DIR)
    !pip install -q -r requirements.txt
else:                                              # running on your own computer
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                             # go to the repo root

sys.path.insert(0, os.getcwd())                    # so "import config" and "from src..." work
!git log -1 --format="Code version: %h  %s"

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import config
from src.files import find_file

base = pd.read_csv(find_file("baselines.csv"))
policy = pd.read_csv(find_file("policy_results.csv"))
budgets = pd.read_csv(find_file("budgets.csv"))
try:
    offload = pd.read_csv(find_file("offload.csv"))
except FileNotFoundError:
    offload = None
    print("offload.csv not found (M8 skipped) -> speed of offloading is only estimated")
base["name"] = base["model"].str.split("/").str[-1]

## Table 1 · OLMoE vs dense models

In [ ]:
t1 = base.set_index("name")[["total_params_B", "active_params_B", "GFLOPs_per_token",
                             "wikitext_perplexity", "code_perplexity", "peak_GPU_GB",
                             "wikitext_prefill_tok_per_s", "wikitext_prefill_J_per_tok",
                             "decode_tok_per_s", "decode_J_per_tok"]]
if offload is not None:
    for _, r in offload.iterrows():
        t1.loc[f"OLMoE offloaded K={r.slots}", ["peak_GPU_GB", "decode_tok_per_s", "decode_J_per_tok"]] = \
            [r.peak_GPU_GB, r.decode_tok_per_s, r.decode_J_per_tok]
t1.round(3).T

## Table 2 · Cache policies (hit rate and waiting time per token)

In [ ]:
t2 = policy.pivot_table(index=["dataset", "slots"], columns="variant",
                        values=["hit_rate", "stall_ms_per_token"], sort=False).round(3)
t2

## Table 3 · Equal vs per-layer budgets

In [ ]:
budgets.round(3)

## Figure · Hit rate vs cache size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, dataset in zip(axes, ["wikitext", "code"]):
    for variant, g in policy[policy.dataset == dataset].groupby("variant", sort=False):
        ax.plot(100 * g.slots / 64, g.hit_rate, marker="o", label=variant)
    ax.set(title=dataset, xlabel="cache size (% of experts per layer)", ylabel="hit rate")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(config.RESULTS_DIR, "M9_hit_rate.png"), dpi=130); plt.show()

## Headline numbers for the report

In [ ]:
names = base.set_index("name")
moe, dense7 = names.loc["OLMoE-1B-7B-0924"], names.loc["OLMo-7B-0724-hf"]
print(f"Compute : OLMoE needs {dense7.GFLOPs_per_token / moe.GFLOPs_per_token:.1f}x fewer FLOPs per token than dense 7B")
print(f"Energy  : {dense7.wikitext_prefill_J_per_tok / moe.wikitext_prefill_J_per_tok:.1f}x less energy per token (prefill)")
print(f"Quality : perplexity OLMoE {moe.wikitext_perplexity:.2f} vs dense 7B {dense7.wikitext_perplexity:.2f} "
      f"vs dense 1B {names.filter(like='OLMo-1B', axis=0).wikitext_perplexity.item():.2f}")
if offload is not None:
    r = offload.iloc[0]
    print(f"Memory  : offloaded OLMoE (K={r.slots}) peaks at {r.peak_GPU_GB:.1f} GB vs dense 7B {dense7.peak_GPU_GB:.1f} GB")

for dataset in ["wikitext", "code"]:
    p = policy[(policy.dataset == dataset) & (policy.slots == 16)].set_index("variant")
    best = p.drop(["Belady", "NoCache"]).stall_ms_per_token.idxmin()
    print(f"\n{dataset}, 16 slots/layer: best real policy = {best}")
    print(f"  waiting per token: NoCache {p.loc['NoCache'].stall_ms_per_token:.0f} ms -> "
          f"{best} {p.loc[best].stall_ms_per_token:.0f} ms (Oracle {p.loc['Belady'].stall_ms_per_token:.0f} ms)")

## Limitations to state in the report
- Stall times in Table 2 are **estimates**: misses × measured copy time (M3). M8 checks the miss
  counts on real hardware, but not every effect (e.g. overlap of copies with compute).
- Offloaded OLMoE is slower per token than a model that fits fully on the GPU.
- Results are for batch size 1 and 2 datasets.